In [2]:
# from dotenv import load_dotenv
# _ = load_dotenv()
import os
from dotenv import load_dotenv
_ = load_dotenv()

In [19]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-4o-mini",
    streaming=True,
    api_key=os.environ.get("OPEN_AI_KEY")
)

In [12]:
from langchain_core.tools import tool

@tool
def calculate(what: str) -> str:
    """Evaluate a mathematical expression."""
    return str(eval(what, {"__builtins__": {}}, {}))

tools = [calculate]

In [14]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
import operator

class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

In [15]:
class Agent:

    def __init__(self, model, tools, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_openai)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges(
            "llm",
            self.exists_action,
            {True: "action", False: END}
        )
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile()
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def call_openai(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            if not t['name'] in self.tools:      # check for bad tool name from LLM
                print("\n ....bad tool name....")
                result = "bad tool name, retry"  # instruct LLM to retry if bad
            else:
                result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

In [20]:
abot = Agent(model, tools, system="You are a math assistant.")

In [22]:
input_data = {
    "messages": [HumanMessage(content="What is 45 * 12?")]
}

# for event in abot.graph.stream(input_data, stream_mode="values"):
#     print(event)

for event in abot.graph.stream(input_data, stream_mode="values"):
    if "messages" in event:
        last_msg = event["messages"][-1]
        print(type(last_msg).__name__, ":", last_msg.content)

HumanMessage : What is 45 * 12?
AIMessage : 
Calling: {'name': 'calculate', 'args': {'what': '45 * 12'}, 'id': 'call_GQl5g2LCqHeMCjDcB9g3JSiR', 'type': 'tool_call'}
Back to the model!
ToolMessage : 540
AIMessage : The result of \( 45 \times 12 \) is \( 540 \).
